In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    SingleStateAnsatz,create_single_machine,\
    create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,\
    NESFermionHopRule,compute_qgt,sampler_info,\
    create_machine_matrix_stable,create_single_machine_gauge_fixed,create_machine_max_stable,\
    NESTotalAnsatz_stable,create_machine_stable,NES_loss_energy_stable,nes_vmc_gradient_stable,\
    make_grad_fn,make_qgt_fn
import optax
from scipy.sparse.linalg import eigsh
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time
import itertools
from pyscf import gto, scf, fci
import numpy as np



jnp.set_printoptions(
    linewidth=9999,
    threshold=jnp.inf,
    precision=8,
    suppress=False,
)


In [ ]:
from LiH import SINGLE_SIZE,ha,hi,hi_ext,ext_edges,single_edges_full,K,Hatree_Fock

In [ ]:
import logging
# 日志配置
logger = logging.getLogger('NES_VMC_K4')
logger.setLevel(logging.INFO)
# 阻止日志向上传播
logger.propagate = False
# 清除所有旧handler，防止重复打印
logger.handlers.clear()

# 自定义日志格式：只打印内容，不带等级、logger名
simple_formatter = logging.Formatter("%(message)s", datefmt="%H:%M:%S")

# 1. 文件输出处理器
file_handler = logging.FileHandler("nes_vmc_0705_K4_LiH_STO-3G.log", mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
file_handler.setLevel(logging.INFO)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
console_handler.setLevel(logging.INFO)
logger.addHandler(console_handler)

print('库导入完成')

In [ ]:

nes_rule = NESFermionHopRule(
    edges=ext_edges,
    K=K,
    single_size=SINGLE_SIZE,
)

In [ ]:
import time
import jax
import jax.numpy as jnp
import optax
from NES_VMC import compute_qgt_fixed,\
    create_single_machine_gauge_fixed,NESTotalAnsatz_gauge_stable,create_machine_gauge_stable,\
        create_machine_matrix_gauge_stable,create_machine_max_gauge_stable

# ====================== 超参统一配置 ======================
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER = 5
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = True
clip_norm = 20.0        # 全局梯度L2上限，QML推荐1~2
lr = 0.1
qgt_diag_shift = 0.1  # 上调正则，抑制QGT梯度爆炸

# ====================== 模型初始化 ======================
from NES_VMC import (
    NESTotalAnsatz_stable,
    create_gauge_fixed_total_machines,
    create_single_machine_gauge_fixed,
    make_grad_fn,
    make_qgt_fn,
)

# ====================== 模型初始化 ======================
total_ansatz = NESTotalAnsatz_stable(
    n_spin_orbitals=SINGLE_SIZE,
    n_states=K,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)

(
    total_machine,
    total_matrix_machine,
    total_max_machine,
    total_graphdef,
    total_params,
) = create_gauge_fixed_total_machines(
    total_ansatz,
    Hatree_Fock,
)

single_machine_list = [
    create_single_machine_gauge_fixed(ansatz, Hatree_Fock)[0]
    for ansatz in total_ansatz.single_ansatz_list
]
    
grad_fn = make_grad_fn(
    ha,
    total_matrix_machine,
    total_max_machine,
    total_machine,
    single_machine_list,
)
qgt_fn = make_qgt_fn(total_machine)

nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=SWEEP_SIZE
)


eigvals, eigvecs = eigsh(ha.to_sparse(), k=K, which="SA", tol=1e-10)
# ====================== 优化器：梯度裁剪 + SGD ======================
# chain顺序：先裁剪梯度，再SGD更新
optimizer = optax.chain(
    optax.clip_by_global_norm(clip_norm),
    optax.sgd(learning_rate=lr)
)
opt_state = optimizer.init(total_params)


sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ====================== 训练历史（新增Ψ矩阵条件数监控） ======================
history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'energy_3st': [],
    'loss': [],
    'params': [],
    'E_Lmatrix': [],
    'samples': [],
    'log_Psi_mean': [],
    'log_Psi_min': [],
    'log_Psi_max': [],
    'grad_norm_raw': [],       # QGT前原始梯度
    'grad_norm_natural': [],   # QGT自然梯度（裁剪前）
    'grad_norm_clipped': [],   # 裁剪后真实梯度（≤clip_norm）
    'psi_cond': [],           # 新增：波函数矩阵条件数
}

logger.info("\n" + "="*60)
logger.info(f"开始多链 NES-VMC 训练 | 使用{'自然' if Natural_Grad else '原始'}梯度")
logger.info("="*60)
logger.info(f"精确CAS基准：基态={eigvals[0]:.8f} Ha | 1激发={eigvals[1]:.8f} Ha|2激发={eigvals[2]:.8f} Ha|3激发={eigvals[3]:.8f} Ha|")
logger.info(f"理论 Loss 上限：{sum(eigvals[0:K]):.8f} ")
logger.info(f"超参：clip_norm={clip_norm}, lr={lr}, QGT diag_shift={qgt_diag_shift}")

start_time = time.time()
for step in range(N_ITER):
    # 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN
    )
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)
    
    grad_raw, loss_mean, E_L_mean, aux = grad_fn(total_params, x_batch)
    
    grad_raw_flat, unravel_fn = ravel_pytree(grad_raw)
    grad_norm_raw = jnp.linalg.norm(grad_raw_flat)
    grad_update = grad_raw

    # ========== 异常提前拦截，防止崩溃卡死 ==========
    has_nan_grad = jnp.any(jnp.isnan(grad_raw_flat))
    grad_explode = grad_norm_raw > 5000.0
    if has_nan_grad or grad_explode:
        logger.warning(f"【Step {step} 告警】梯度异常！nan={has_nan_grad}, raw_grad_norm={grad_norm_raw:.2f}")

    # 2. QGT自然梯度预条件
    if Natural_Grad:        
        qgt_reg_mat = qgt_fn(total_params, x_batch, qgt_diag_shift)
        ng_flat = jnp.linalg.solve(qgt_reg_mat, grad_raw_flat)
        

        grad_update = unravel_fn(ng_flat)
        grad_norm_natural = jnp.linalg.norm(ng_flat)
    else:
        grad_norm_natural = grad_norm_raw

    # 3. 梯度裁剪（optimizer.update内部自动执行）
    updates, opt_state = optimizer.update(grad_update, opt_state, total_params)
    # 单独计算裁剪后梯度范数用于监控
    clip_transform = optax.clip_by_global_norm(clip_norm)
    clipped_grad, _ = clip_transform.update(grad_update, opt_state[0], total_params)
    clipped_flat, _ = ravel_pytree(clipped_grad)
    grad_norm_clipped = jnp.linalg.norm(clipped_flat)
    
    # 如果梯度范数为0 则结束迭代
    if grad_norm_clipped == 0.0:
        logger.info(f"【Step {step} 告警】梯度范数为0，结束迭代")
        break

    # 参数更新
    total_params = optax.apply_updates(total_params, updates)

    # ====================== 新增：计算Ψ矩阵条件数 ======================
    # 取单批次样本计算波函数矩阵，用第一组组态做代表
    x_single = x_batch[0:1, ...]
    psi_mat = total_matrix_machine(total_params, x_single)[0]  # 取出N×N波函数矩阵
    psi_cond = jnp.linalg.cond(psi_mat)

    # 波函数标量输出
    log_Psi_batch = total_machine(total_params, x_batch)
    #eig_vals, eig_vecs = jnp.linalg.eig(E_L_mean)
    
    
    eig_vals, eig_vecs = jnp.linalg.eig(E_L_mean)
    sort_idx = jnp.argsort(eig_vals.real)
    eig_vals, eig_vecs = eig_vals[sort_idx], eig_vecs[:, sort_idx]

    # 记录历史
    history['step'].append(step)
    history['loss'].append(loss_mean)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['energy_3st'].append(eig_vals[3])
    history['params'].append(total_params)
    history['grad_norm_raw'].append(grad_norm_raw)
    history['grad_norm_natural'].append(grad_norm_natural)
    history['grad_norm_clipped'].append(grad_norm_clipped)
    history['psi_cond'].append(psi_cond)  # 保存条件数

    # 打印日志，新增Ψ条件数输出
    if step % 1 == 0 or step == N_ITER - 1:
        logger.info(f"[Step {step:3d}] logΨ: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")
        logger.info(f"梯度监控 | raw={grad_norm_raw:.4f} | natural={grad_norm_natural:.4f} | clipped={grad_norm_clipped:.4f}(上限{clip_norm})")
        logger.info(f"Ψ矩阵条件数 cond(Ψ) = {psi_cond:.2e}")
        logger.info(f"Loss={loss_mean:.6f} | E0={eig_vals[0]:.8f} | E1={eig_vals[1]:.8f}|E2={eig_vals[2]:.8f}|E3={eig_vals[3]:.8f}")
        logger.info("#-----------------------------------------#")

end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
print("\n" + "="*60)
print("训练完成!")
print("="*60)

In [ ]:
import pickle
import os
os.makedirs('./data', exist_ok=True)

if Natural_Grad == True:
    print('保存自然梯度历史记录')
    with open('./data/history_natural_gradient_LiH_molecule_K4.pkl', 'wb') as f:
        pickle.dump(history, f)
else:
    with open('./data/history_plain_gradient_LiH_molecule_K4.pkl', 'wb') as f:
        pickle.dump(history, f)

print("保存成功！")

In [ ]:
import pickle
history = pickle.load(open('./data/history_natural_gradient_LiH_molecule_K4.pkl', 'rb'))

In [ ]:
import matplotlib.pyplot as plt

# 创建子图
fig, axs = plt.subplots(3, 4, figsize=(16, 9))
fig.suptitle('Natural Excited State-VMC for LiH Molecule K=4 (cc-pVDZ)')

# 第一行：能量
axs[0, 0].plot(history['energy_0st'][:53], color='orange', label='NES-VMC')
axs[0, 0].hlines(eigvals[0], 0, len(history['energy_0st'][:53]), linestyle='--', color='red', label='Exact')
axs[0, 0].set_title('0st Energy')
axs[0, 0].set_ylabel('energy')
axs[0, 0].set_xlabel('step')
axs[0, 0].legend()

axs[0, 1].plot(history['energy_1st'][:53], color='orange', label='NES-VMC')
axs[0, 1].hlines(eigvals[1], 0, len(history['energy_1st'][:53]), linestyle='--', color='red', label='Exact')
axs[0, 1].set_title('1st Energy')
axs[0, 1].set_ylabel('energy')
axs[0, 1].set_xlabel('step')
axs[0, 1].legend()

axs[0, 2].plot(history['energy_2st'][:53], color='orange', label='NES-VMC')
axs[0, 2].hlines(eigvals[2], 0, len(history['energy_2st'][:53]), linestyle='--', color='red', label='FCI')
axs[0, 2].set_title('2st Energy')
axs[0, 2].set_ylabel('energy')
axs[0, 2].set_xlabel('step')
axs[0, 2].legend()

axs[0, 3].plot(history['energy_3st'][:53], color='orange', label='NES-VMC')
axs[0, 3].hlines(eigvals[3], 0, len(history['energy_3st'][:53]), linestyle='--', color='red', label='FCI')
axs[0, 3].set_title('3st Energy')
axs[0, 3].set_ylabel('energy')
axs[0, 3].set_xlabel('step')
axs[0, 3].legend()

# # 第二行：能量误差
# axs[1, 0].plot(history['energy_0st'] - eigvals[0], color='orange')
# axs[1, 0].set_title('0st Energy Error')
# axs[1, 0].set_xlabel('step')
# axs[1, 0].set_ylabel('energy error')

# axs[1, 1].plot(history['energy_1st'] - E_fcis[1], color='orange')
# axs[1, 1].set_title('1st Energy Error')
# axs[1, 1].set_xlabel('step')
# axs[1, 1].set_ylabel('energy error')

# axs[1, 2].plot(history['energy_2st'] - E_fcis[2], color='orange')
# axs[1, 2].set_title('2st Energy Error')
# axs[1, 2].set_xlabel('step')
# axs[1, 2].set_ylabel('energy error')

# axs[1, 3].plot(history['energy_3st'] - E_fcis[3], color='orange')
# axs[1, 3].set_title('3st Energy Error')
# axs[1, 3].set_xlabel('step')
# axs[1, 3].set_ylabel('energy error')

# # 第三行：loss 和 grad_norm
# axs[2, 0].plot(history['loss'], color='orange')
# axs[2, 0].set_title('loss')
# axs[2, 0].set_xlabel('step')
# axs[2, 0].set_ylabel('loss')

# axs[2, 1].plot(history['grad_norm'], color='orange')
# axs[2, 1].set_title('grad_norm')
# axs[2, 1].set_xlabel('step')
# axs[2, 1].set_ylabel('grad_norm')

axs[2, 2].plot(history['energy_0st'][30:53], color='blue')
axs[2, 2].plot(history['energy_1st'][30:53], color='orange')
axs[2, 2].plot(history['energy_2st'][30:53], color='green')
axs[2, 2].plot(history['energy_3st'][30:53], color='purple')
axs[2, 2].hlines(eigvals[0], 0, len(history['energy_0st'][30:53]), linestyle='--', color='red', label='FCI 0')
axs[2, 2].hlines(eigvals[1], 0, len(history['energy_1st'][30:53]), linestyle='--', color='red', label='FCI 1')
axs[2, 2].hlines(eigvals[2], 0, len(history['energy_2st'][30:53]), linestyle='--', color='red', label='FCI 2')
axs[2, 2].hlines(eigvals[3], 0, len(history['energy_3st'][30:53]), linestyle='--', color='red', label='FCI 3')
axs[2, 2].set_title('Energy Comparison')
axs[2, 2].set_xlabel('step')
axs[2, 2].set_ylabel('energy')
axs[2, 2].legend()

# # 隐藏未使用的子图
# axs[2, 3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 创建 2行1列 的子图
fig, axs = plt.subplots(1, 3, figsize=(12, 6))
fig.suptitle('NES-VMC for $H$ Atom K=3 under cc-pVDZ Basis ')


history_trunc = {k: v[:53] for k, v in history.items()}
# 第一个子图
axs[0].plot(history_trunc['energy_0st'],label='NES-VMC 0st Energy')
axs[0].hlines(E_fcis[0],0,len(history_trunc['energy_0st']),linestyle='--',color='red',label='0st Energy FCI')
axs[0].plot(history_trunc['energy_2st'],label='NES-VMC 1st Energy')
axs[0].hlines(E_fcis[1],0,len(history_trunc['energy_1st']),linestyle='--',color='red',label='1st Energy FCI')
axs[0].plot(history['energy_3st'],label='NES-VMC 2st Energy')   
axs[0].hlines(E_fcis[2],0,len(history['energy_2st']),linestyle='--',color='red',label='2st Energy FCI')
axs[0].plot(history['energy_3st'],label='NES-VMC 3st Energy')
axs[0].hlines(E_fcis[3],0,len(history['energy_3st']),linestyle='--',color='red',label='3st Energy FCI')

axs[0].set_title('K=4 Energy Levels')
axs[0].set_ylabel('Energy (Ha)')
axs[0].set_xlabel('step')
axs[0].legend()



# 第二个子图
axs[1].plot(history['loss'])
axs[1].set_title('loss')
axs[1].set_xlabel('step')
axs[1].set_ylabel('energy')

# 第三个子图
axs[2].plot(history['grad_norm_natural'])
axs[2].set_title('Natural Gradient Norm')
axs[2].set_xlabel('step')
axs[2].set_ylabel('grad_norm')

plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 创建 2行1列 的子图
fig, axs = plt.subplots(1,2, figsize=(12, 6))
fig.suptitle('NES-VMC for $H$ Atom K=3 under cc-pVDZ Basis ')


history_trunc = {k: v[:53] for k, v in history.items()}
# 第一个子图
axs[0].plot(history_trunc['energy_0st'],label='NES-VMC 0st Energy')
axs[0].hlines(eig_vals[0],0,len(history_trunc['energy_0st']),linestyle='--',color='red',label='0st Energy FCI')
# axs[0].plot(history_trunc['energy_2st'],label='NES-VMC 1st Energy')
# axs[0].hlines(ei[1],0,len(history_trunc['energy_1st']),linestyle='--',color='red',label='1st Energy FCI')
# axs[0].plot(history['energy_3st'],label='NES-VMC 2st Energy')   
# axs[0].hlines(E_fcis[2],0,len(history['energy_2st']),linestyle='--',color='red',label='2st Energy FCI')
# axs[0].plot(history['energy_3st'],label='NES-VMC 3st Energy')
# axs[0].hlines(E_fcis[3],0,len(history['energy_3st']),linestyle='--',color='red',label='3st Energy FCI')

axs[0].set_title('K=4 Energy Levels')
axs[0].set_ylabel('Energy (Ha)')
axs[0].set_xlabel('step')
axs[0].legend()


plt.tight_layout()  # 自动调整间距
plt.show()

In [ ]:
import pickle
with open('./data/history_natural_gradient_LiH_molecule_K4.pkl', 'rb') as f:
    history = pickle.load(f)

In [ ]:
params = history['params'][0]
samples = history['samples'][0]



In [12]:
total_machine(params,samples)

Array([4.62748082-6.34713254e-01j, 5.81026932+2.35708103e-01j, 5.79498798+8.63560014e-01j, 3.98733346+2.30520703e+00j, 6.36215507-5.99465223e-01j, 3.9976878 +1.03442089e+00j, 5.5913004 +2.88435162e+00j, 5.07843053+2.93005018e+00j, 5.66731659-2.03412644e+00j, 6.32144914+1.88606184e+00j, 6.39588503-3.42932216e-01j, 6.37292192+8.90577784e-01j, 6.67022804+2.49489354e+00j, 5.79499336+4.10504799e-01j, 6.32144914+1.88606184e+00j, 5.49252761-1.41497428e+00j, 6.08963257+2.81638816e+00j, 5.91588625-8.80905155e-01j, 4.68734604-1.82115268e+00j, 6.3692903 -2.99706955e+00j, 5.95433518-3.01769930e+00j, 5.27490319+2.55247268e+00j, 5.56229952-2.27642609e+00j, 6.13715195-1.36350546e+00j, 5.55361817+1.86285885e+00j, 6.32920913+3.11053387e+00j, 6.8336639 +2.07898239e+00j, 5.60618969+7.68575409e-01j, 6.34016915+9.52338169e-01j, 5.58837593-9.69531289e-01j, 5.27484301-1.36008701e+00j, 5.50828068-1.80538437e+00j, 4.90531944+1.89776656e+00j, 5.42395659+2.12809501e+00j, 4.65701987-2.46940868e+00j, 5.76600822-2.

In [13]:
total_matrix_machine(params,samples)


Array([[[-0.99181115+4.45832756e-01j, -2.18391064+2.20743340e-01j, -1.03534748+3.21360371e-01j, -2.25309067+6.90005804e-01j],
        [-1.62195483+1.39554795e+00j,  0.        -7.33103066e-01j, -2.70081475+2.71784600e-02j, -1.81623138-5.72888677e-01j],
        [-1.99684521+1.96386968e-02j, -2.08282155-1.37504267e-01j, -1.25987232+3.84399055e-01j, -2.52516683-3.18646042e-01j],
        [-0.47395453+8.55530606e-01j, -2.04060907+7.32396533e-01j, -1.89322909+2.77996914e-01j, -2.19396645+2.34169965e-01j]],

       [[-2.05599029+1.13735769e+00j,  0.        -5.32663421e-01j, -0.42548731-1.92961985e-01j, -1.81233212-2.64192748e-01j],
        [-1.084677  +2.71064082e+00j, -0.3567861 +2.07099437e+00j, -1.89782173+7.70743215e-01j, -1.9055736 -4.78952914e-02j],
        [-1.54336881+4.09689015e-01j, -1.78658134+4.22927779e-01j, -2.17668121-1.49317236e-01j, -0.36471377-1.65573513e+00j],
        [-0.87823875-2.40980336e-02j, -2.12667348+3.99251688e-01j, -1.18806915-2.49894958e-01j, -1.80585327-1.188695